# Cholinergic ligand latent space: conformer clouds + attention AE

Small exploratory project for seven related cholinergic ligands: acetylcholine, choline, carbachol, bethanechol, muscarine, nicotine, and atropine.

The key change is that each data point is now one minimized conformer, not one ligand. Morgan fingerprints and 2D RDKit descriptors are repeated across conformers of the same ligand, while force-field energy and 3D shape descriptors vary across conformers. This gives the AE a chance to learn common conformational/property axes shared across ligands, and lets us inspect clouds rather than seven isolated points.

Important limitation: this is still a small exploratory dataset. It can suggest motifs and axes, but it is not a statistically defensible chemical manifold by itself.

The “simulation” here is fast RDKit conformer sampling plus MMFF/UFF force-field minimization. It is not full molecular dynamics.


In [ ]:
import sys
from pathlib import Path

try:
    import rdkit
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "RDKit is required for this notebook. Install it in the project venv with:\n"
        "  .venv/bin/python -m pip install rdkit\n"
        "Then restart the notebook kernel."
    ) from exc

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, Crippen, Lipinski, rdMolDescriptors
from rdkit.Chem import rdFingerprintGenerator

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "lss").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SEED = 20260707
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("RDKit", rdkit.__version__, "device", DEVICE)

## Ligands

For quaternary ammonium ligands, the parent cation is used without arbitrary chloride/salt counterions. Otherwise the latent space would partly separate by counterion choice rather than by the ligand itself.

In [ ]:
ligands = pd.DataFrame([
    {
        "name": "Acetylcholine",
        "smiles": "CC(=O)OCC[N+](C)(C)C",
        "class": "endogenous agonist",
        "primary_target": "muscarinic + nicotinic",
    },
    {
        "name": "Choline",
        "smiles": "C[N+](C)(C)CCO",
        "class": "precursor",
        "primary_target": "precursor",
    },
    {
        "name": "Carbachol",
        "smiles": "NC(=O)OCC[N+](C)(C)C",
        "class": "agonist",
        "primary_target": "muscarinic + nicotinic",
    },
    {
        "name": "Bethanechol",
        "smiles": "NC(=O)OC(C)C[N+](C)(C)C",
        "class": "agonist",
        "primary_target": "muscarinic",
    },
    {
        "name": "Muscarine",
        "smiles": "O[C@@H]1C[C@H](O[C@H]1C)C[N+](C)(C)C",
        "class": "agonist",
        "primary_target": "muscarinic",
    },
    {
        "name": "Nicotine",
        "smiles": "CN1CCC[C@H]1c2cccnc2",
        "class": "agonist",
        "primary_target": "nicotinic",
    },
    {
        "name": "Atropine",
        "smiles": "CN1C2CCC1CC(C2)OC(=O)C(CO)c3ccccc3",
        "class": "antagonist",
        "primary_target": "muscarinic",
    },
])

ligands["mol"] = ligands["smiles"].map(Chem.MolFromSmiles)
bad = ligands[ligands["mol"].isna()]
assert bad.empty, bad[["name", "smiles"]]

ligands["canonical_smiles"] = ligands["mol"].map(lambda mol: Chem.MolToSmiles(mol, isomericSmiles=True))
display(ligands.drop(columns="mol"))

## Fast conformer simulation

This generates several 3D conformers per molecule, minimizes each with MMFF94s when available and UFF otherwise, then keeps one row per conformer. The resulting table is the simulation-like dataset used by the AE.


In [ ]:
N_CONFS = 1000
MAX_ITERS = 250


def safe_float(fn, default=np.nan):
    try:
        value = float(fn())
        return value if np.isfinite(value) else default
    except Exception:
        return default


def simulate_conformers(mol, *, ligand_name: str, seed: int = SEED, n_confs: int = N_CONFS) -> list[dict]:
    mol_h = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = int(seed)
    params.useSmallRingTorsions = True
    params.pruneRmsThresh = 0.25
    conf_ids = list(AllChem.EmbedMultipleConfs(mol_h, numConfs=int(n_confs), params=params))
    if not conf_ids:
        return [{"name": ligand_name, "conformer_id": -1, "force_field": "failed"}]

    force_field = "MMFF94s" if AllChem.MMFFHasAllMoleculeParams(mol_h) else "UFF"
    mmff_props = AllChem.MMFFGetMoleculeProperties(mol_h, mmffVariant="MMFF94s") if force_field == "MMFF94s" else None

    rows = []
    for local_idx, conf_id in enumerate(conf_ids):
        conf_id = int(conf_id)
        if force_field == "MMFF94s":
            ff = AllChem.MMFFGetMoleculeForceField(mol_h, mmff_props, confId=conf_id)
        else:
            ff = AllChem.UFFGetMoleculeForceField(mol_h, confId=conf_id)
        status = int(ff.Minimize(maxIts=int(MAX_ITERS)))
        energy = float(ff.CalcEnergy())
        rows.append({
            "name": ligand_name,
            "conformer_id": int(local_idx),
            "rdkit_conf_id": conf_id,
            "force_field": force_field,
            "ff_converged": int(status == 0),
            "ff_energy": energy,
            "radius_gyration": safe_float(lambda conf_id=conf_id: rdMolDescriptors.CalcRadiusOfGyration(mol_h, confId=conf_id)),
            "asphericity": safe_float(lambda conf_id=conf_id: rdMolDescriptors.CalcAsphericity(mol_h, confId=conf_id)),
            "inertial_shape_factor": safe_float(lambda conf_id=conf_id: rdMolDescriptors.CalcInertialShapeFactor(mol_h, confId=conf_id)),
            "npr1": safe_float(lambda conf_id=conf_id: rdMolDescriptors.CalcNPR1(mol_h, confId=conf_id)),
            "npr2": safe_float(lambda conf_id=conf_id: rdMolDescriptors.CalcNPR2(mol_h, confId=conf_id)),
        })

    energies = np.asarray([row["ff_energy"] for row in rows], dtype=float)
    min_energy = float(np.nanmin(energies))
    for row in rows:
        row["relative_ff_energy"] = float(row["ff_energy"] - min_energy)
        row["n_conformers_for_ligand"] = int(len(rows))
    return rows


conformer_rows = []
for idx, row in ligands.reset_index(drop=True).iterrows():
    conformer_rows.extend(simulate_conformers(row.mol, ligand_name=row["name"], seed=SEED + idx))

conformer_df = pd.DataFrame(conformer_rows)
conformer_df["name"] = conformer_df["name"].astype(str)
display(conformer_df.groupby("name", as_index=False).agg(
    n_conformers=("conformer_id", "count"),
    converged_fraction=("ff_converged", "mean"),
    min_energy=("ff_energy", "min"),
    energy_std=("ff_energy", "std"),
))
display(conformer_df.head())


## Morgan fingerprint + physicochemical descriptor matrix

Each conformer row receives the same ligand-level Morgan fingerprint and 2D descriptors, plus conformer-specific minimized energy and 3D shape descriptors.


In [ ]:
FP_SIZE = 256
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_SIZE)
mol_by_name = dict(zip(ligands["name"].astype(str), ligands["mol"]))


def morgan_bits(mol) -> np.ndarray:
    arr = np.zeros((FP_SIZE,), dtype=np.float32)
    fp = morgan.GetFingerprint(mol)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def descriptor_row(name: str, mol) -> dict:
    has_cationic_n = any(atom.GetSymbol() == "N" and atom.GetFormalCharge() > 0 for atom in mol.GetAtoms())
    has_aromatic_atom = any(atom.GetIsAromatic() for atom in mol.GetAtoms())
    ester = Chem.MolFromSmarts("[CX3](=O)[OX2][#6]")
    carbamate = Chem.MolFromSmarts("[NX3][CX3](=O)[OX2][#6]")
    return {
        "name": name,
        "mol_wt": Descriptors.MolWt(mol),
        "heavy_atoms": Descriptors.HeavyAtomCount(mol),
        "formal_charge": Chem.GetFormalCharge(mol),
        "h_donors": Lipinski.NumHDonors(mol),
        "h_acceptors": Lipinski.NumHAcceptors(mol),
        "tpsa": rdMolDescriptors.CalcTPSA(mol),
        "logp": Crippen.MolLogP(mol),
        "rotatable_bonds": Lipinski.NumRotatableBonds(mol),
        "ring_count": Lipinski.RingCount(mol),
        "aromatic_rings": Lipinski.NumAromaticRings(mol),
        "fraction_csp3": rdMolDescriptors.CalcFractionCSP3(mol),
        "has_cationic_n": float(has_cationic_n),
        "has_aromatic_atom": float(has_aromatic_atom),
        "has_ester": float(mol.HasSubstructMatch(ester)),
        "has_carbamate": float(mol.HasSubstructMatch(carbamate)),
    }


descriptor_df = pd.DataFrame([descriptor_row(name, mol) for name, mol in mol_by_name.items()])
descriptor_df["name"] = descriptor_df["name"].astype(str)

feature_df = (
    ligands[["name", "class", "primary_target", "canonical_smiles"]]
    .assign(name=lambda df: df["name"].astype(str))
    .merge(descriptor_df, on="name", validate="one_to_one")
    .merge(conformer_df, on="name", validate="one_to_many")
    .sort_values(["name", "conformer_id"])
    .reset_index(drop=True)
)

numeric_cols = [
    "mol_wt", "heavy_atoms", "formal_charge", "h_donors", "h_acceptors", "tpsa", "logp",
    "rotatable_bonds", "ring_count", "aromatic_rings", "fraction_csp3",
    "has_cationic_n", "has_aromatic_atom", "has_ester", "has_carbamate",
    "ff_converged", "ff_energy", "relative_ff_energy",
    "radius_gyration", "asphericity", "inertial_shape_factor", "npr1", "npr2",
]
feature_df[numeric_cols] = feature_df[numeric_cols].replace([np.inf, -np.inf], np.nan)
feature_df[numeric_cols] = feature_df[numeric_cols].fillna(feature_df[numeric_cols].median(numeric_only=True))

fp_matrix = np.vstack([morgan_bits(mol_by_name[name]) for name in feature_df["name"]]).astype(np.float32)
num_raw = feature_df[numeric_cols].to_numpy(dtype=np.float32)
num_mean = num_raw.mean(axis=0, keepdims=True)
num_std = num_raw.std(axis=0, keepdims=True)
num_std[num_std < 1e-8] = 1.0
num_scaled = (num_raw - num_mean) / num_std

X = np.concatenate([fp_matrix, num_scaled], axis=1).astype(np.float32)
type_ids = np.concatenate([np.zeros(FP_SIZE, dtype=np.int64), np.ones(len(numeric_cols), dtype=np.int64)])

display(feature_df[["name", "conformer_id", "class", "primary_target", "ff_energy", "relative_ff_energy", "radius_gyration", "ring_count", "logp"]].head(12))
print("feature matrix", X.shape, "fingerprint bits", FP_SIZE, "numeric descriptors", len(numeric_cols))


## Attention autoencoder

Each Morgan bit or descriptor is treated as a feature token. The encoder uses self-attention over tokens, then learned latent query tokens compress each conformer sample to a 2D latent vector. The decoder cross-attends feature queries to the latent tokens and reconstructs the fingerprint bits plus standardized descriptors.


In [ ]:
class FeatureAttentionAE(nn.Module):
    def __init__(self, n_features: int, type_ids: np.ndarray, *, hidden: int = 64, heads: int = 4, latent_dim: int = 2, latent_tokens: int = 4, layers: int = 2):
        super().__init__()
        self.n_features = int(n_features)
        self.hidden = int(hidden)
        self.latent_tokens = int(latent_tokens)
        self.feature_embed = nn.Embedding(n_features, hidden)
        self.type_embed = nn.Embedding(2, hidden)
        self.value_proj = nn.Linear(1, hidden)
        self.register_buffer("feature_ids", torch.arange(n_features, dtype=torch.long))
        self.register_buffer("type_ids", torch.as_tensor(type_ids, dtype=torch.long))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden,
            nhead=heads,
            dim_feedforward=hidden * 2,
            dropout=0.05,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.latent_queries = nn.Parameter(torch.randn(1, latent_tokens, hidden) * 0.02)
        self.encode_cross = nn.MultiheadAttention(hidden, heads, batch_first=True)
        self.to_z = nn.Sequential(nn.LayerNorm(latent_tokens * hidden), nn.Linear(latent_tokens * hidden, latent_dim))

        self.from_z = nn.Linear(latent_dim, latent_tokens * hidden)
        self.decode_cross = nn.MultiheadAttention(hidden, heads, batch_first=True)
        self.out = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, 1))

    def token_base(self, batch_size: int) -> torch.Tensor:
        base = self.feature_embed(self.feature_ids) + self.type_embed(self.type_ids)
        return base.unsqueeze(0).expand(batch_size, -1, -1)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        tokens = self.token_base(x.size(0)) + self.value_proj(x.unsqueeze(-1))
        h = self.encoder(tokens)
        q = self.latent_queries.expand(x.size(0), -1, -1)
        latent_tokens, _ = self.encode_cross(q, h, h, need_weights=False)
        return self.to_z(latent_tokens.reshape(x.size(0), -1))

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        latent_tokens = self.from_z(z).reshape(z.size(0), self.latent_tokens, self.hidden)
        q = self.token_base(z.size(0))
        decoded, _ = self.decode_cross(q, latent_tokens, latent_tokens, need_weights=False)
        return self.out(decoded).squeeze(-1)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        z = self.encode(x)
        return self.decode(z), z


x_tensor = torch.as_tensor(X, dtype=torch.float32, device=DEVICE)
model = FeatureAttentionAE(X.shape[1], type_ids, hidden=64, heads=4, latent_dim=2, latent_tokens=4, layers=2).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)

history = []
for epoch in range(1, 1501):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    recon, z = model(x_tensor)
    bit_loss = F.binary_cross_entropy_with_logits(recon[:, :FP_SIZE], x_tensor[:, :FP_SIZE])
    num_loss = F.mse_loss(recon[:, FP_SIZE:], x_tensor[:, FP_SIZE:])
    latent_loss = 1e-4 * z.pow(2).mean()
    loss = bit_loss + num_loss + latent_loss
    loss.backward()
    optimizer.step()

    if epoch == 1 or epoch % 100 == 0:
        history.append({"epoch": epoch, "loss": float(loss.detach().cpu()), "bit_bce": float(bit_loss.detach().cpu()), "num_mse": float(num_loss.detach().cpu())})

history_df = pd.DataFrame(history)
display(history_df.tail())

print("Plot: AE reconstruction loss over training epochs")
plt.figure(figsize=(5.2, 3.2))
plt.plot(history_df["epoch"], history_df["loss"], marker="o")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("training loss")
plt.grid(alpha=0.25)
plt.show()


## Latent conformer clouds and reconstruction quality


In [ ]:
model.eval()
with torch.no_grad():
    recon, z = model(x_tensor)
    bit_prob = torch.sigmoid(recon[:, :FP_SIZE]).cpu().numpy()
    num_pred = recon[:, FP_SIZE:].cpu().numpy()
    z_np = z.cpu().numpy()

latent_df = feature_df.copy()
latent_df["z0"] = z_np[:, 0]
latent_df["z1"] = z_np[:, 1]
latent_df["fingerprint_reconstruction_accuracy"] = ((bit_prob > 0.5) == fp_matrix.astype(bool)).mean(axis=1)
latent_df["descriptor_reconstruction_mse"] = ((num_pred - num_scaled) ** 2).mean(axis=1)

display(latent_df.groupby("name", as_index=False).agg(
    n_conformers=("conformer_id", "count"),
    z0_mean=("z0", "mean"),
    z1_mean=("z1", "mean"),
    z0_std=("z0", "std"),
    z1_std=("z1", "std"),
    fp_acc=("fingerprint_reconstruction_accuracy", "mean"),
    desc_mse=("descriptor_reconstruction_mse", "mean"),
).round(4))

class_colors = {
    "endogenous agonist": "tab:blue",
    "precursor": "tab:gray",
    "agonist": "tab:green",
    "antagonist": "tab:red",
}
name_colors = dict(zip(sorted(latent_df["name"].unique()), plt.cm.tab10(np.linspace(0, 1, latent_df["name"].nunique()))))
centroids = latent_df.groupby(["name", "class"], as_index=False).agg(z0=("z0", "mean"), z1=("z1", "mean"))

print("Plot: conformer clouds colored by ligand; large markers are ligand centroids")
fig, ax = plt.subplots(figsize=(6.2, 4.8), constrained_layout=True)
for name, group in latent_df.groupby("name"):
    ax.scatter(group["z0"], group["z1"], s=22, alpha=0.35, color=name_colors[name], linewidth=0)
for _, row in centroids.iterrows():
    ax.scatter(row["z0"], row["z1"], s=115, color=name_colors[row["name"]], edgecolor="black", linewidth=0.8)
    ax.annotate(row["name"], (row["z0"], row["z1"]), xytext=(5, 4), textcoords="offset points", fontsize=9)
ax.set_xlabel("z0")
ax.set_ylabel("z1")
ax.grid(alpha=0.25)
plt.show()

print("Plot: same conformer latent space colored by pharmacology label")
fig, ax = plt.subplots(figsize=(6.2, 4.8), constrained_layout=True)
for cls, group in latent_df.groupby("class"):
    ax.scatter(group["z0"], group["z1"], s=24, label=cls, color=class_colors.get(cls, "black"), alpha=0.45, linewidth=0)
for _, row in centroids.iterrows():
    ax.annotate(row["name"], (row["z0"], row["z1"]), xytext=(5, 4), textcoords="offset points", fontsize=9)
ax.set_xlabel("z0")
ax.set_ylabel("z1")
ax.grid(alpha=0.25)
ax.legend(frameon=False, loc="best")
plt.show()


## What does the latent space correlate with?

Now the correlations are over conformer samples, so conformer-varying properties like relative force-field energy and 3D shape can matter. Ligand-level descriptors are repeated across conformers, so they still mostly explain between-ligand separation.


In [ ]:
def pearson_np(a, b) -> float:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 3 or np.isclose(np.std(a[mask]), 0.0) or np.isclose(np.std(b[mask]), 0.0):
        return np.nan
    return float(np.corrcoef(a[mask], b[mask])[0, 1])


corr_rows = []
for col in numeric_cols:
    r0 = pearson_np(latent_df["z0"], latent_df[col])
    r1 = pearson_np(latent_df["z1"], latent_df[col])
    corr_rows.append({
        "property": col,
        "r_z0": r0,
        "r_z1": r1,
        "max_abs_r": np.nanmax(np.abs([r0, r1])),
        "varies_within_ligand": bool(latent_df.groupby("name")[col].std().fillna(0).gt(1e-8).any()),
    })
corr_df = pd.DataFrame(corr_rows).sort_values("max_abs_r", ascending=False)
display(corr_df.round(3))

top_properties = corr_df.head(4)["property"].tolist()
for prop in top_properties:
    print(f"Plot: conformer latent space colored by {prop}")
    fig, ax = plt.subplots(figsize=(5.8, 4.6), constrained_layout=True)
    sc = ax.scatter(latent_df["z0"], latent_df["z1"], c=latent_df[prop], cmap="viridis", s=28, alpha=0.65, linewidth=0)
    for _, row in centroids.iterrows():
        ax.annotate(row["name"], (row["z0"], row["z1"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
    fig.colorbar(sc, ax=ax, pad=0.01, label=prop)
    ax.set_xlabel("z0")
    ax.set_ylabel("z1")
    ax.grid(alpha=0.25)
    plt.show()


## Does the latent space encode ligand identity or shared conformer variation?

This decomposes latent variance into between-ligand centroid variance and within-ligand conformer-cloud variance. More within-ligand variance means the AE is using conformer-level information; more between-ligand variance means it is mostly separating molecule identities/properties.


In [ ]:
Z = latent_df[["z0", "z1"]].to_numpy(float)
grand = Z.mean(axis=0, keepdims=True)
total_ss = float(((Z - grand) ** 2).sum())
within_ss = 0.0
between_ss = 0.0
for name, group in latent_df.groupby("name"):
    Zi = group[["z0", "z1"]].to_numpy(float)
    ci = Zi.mean(axis=0, keepdims=True)
    within_ss += float(((Zi - ci) ** 2).sum())
    between_ss += float(len(Zi) * ((ci - grand) ** 2).sum())

variance_decomposition = pd.DataFrame([
    {"component": "within ligand conformer clouds", "fraction": within_ss / total_ss},
    {"component": "between ligand centroids", "fraction": between_ss / total_ss},
])
display(variance_decomposition.round(3))

print("Plot: latent variance decomposition into within-ligand and between-ligand components")
fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.bar(variance_decomposition["component"], variance_decomposition["fraction"], color=["tab:purple", "tab:orange"])
ax.set_ylim(0, 1)
ax.set_ylabel("fraction of latent variance")
ax.tick_params(axis="x", rotation=18)
ax.grid(axis="y", alpha=0.25)
plt.show()


## PCA baseline

This is a sanity check. If PCA and the AE tell the same story, the structure probably comes directly from the input descriptors/fingerprints rather than from nonlinear attention structure.


In [ ]:
X_centered = X - X.mean(axis=0, keepdims=True)
u, s, vt = np.linalg.svd(X_centered, full_matrices=False)
pca_xy = u[:, :2] * s[:2]
explained = (s ** 2) / np.sum(s ** 2)
pca_df = latent_df[["name", "class", "conformer_id"]].copy()
pca_df["pc1"] = pca_xy[:, 0]
pca_df["pc2"] = pca_xy[:, 1]
pca_centroids = pca_df.groupby(["name", "class"], as_index=False).agg(pc1=("pc1", "mean"), pc2=("pc2", "mean"))

print("Plot: attention AE conformer latent space, repeated for direct comparison with PCA")
fig, ax = plt.subplots(figsize=(5.8, 4.6), constrained_layout=True)
for name, group in latent_df.groupby("name"):
    ax.scatter(group["z0"], group["z1"], s=22, alpha=0.35, color=name_colors[name], linewidth=0)
for _, row in centroids.iterrows():
    ax.scatter(row["z0"], row["z1"], s=115, color=name_colors[row["name"]], edgecolor="black", linewidth=0.8)
    ax.annotate(row["name"], (row["z0"], row["z1"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("z0")
ax.set_ylabel("z1")
ax.grid(alpha=0.25)
plt.show()

print(f"Plot: PCA baseline on the same feature matrix; PC1+PC2 explain {explained[:2].sum():.1%} variance")
fig, ax = plt.subplots(figsize=(5.8, 4.6), constrained_layout=True)
for name, group in pca_df.groupby("name"):
    ax.scatter(group["pc1"], group["pc2"], s=22, alpha=0.35, color=name_colors[name], linewidth=0)
for _, row in pca_centroids.iterrows():
    ax.scatter(row["pc1"], row["pc2"], s=115, color=name_colors[row["name"]], edgecolor="black", linewidth=0.8)
    ax.annotate(row["name"], (row["pc1"], row["pc2"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(alpha=0.25)
plt.show()


## Interpretation template

Use this only after running the notebook:

- If the conformer clouds of acetylcholine/choline/carbachol/bethanechol overlap, the AE is picking up the shared choline-like cationic scaffold rather than strict ligand identity.
- If clouds are compact and well separated by ligand, the latent space mostly encodes ligand identity/fingerprint structure.
- If within-ligand variance is substantial, the latent space uses conformer-level energy/shape information.
- If nicotine separates from the quaternary ligands, the latent space is likely picking up tertiary amine/aromaticity/logP rather than cholinergic activity in general.
- If atropine separates strongly, that is expected: it is larger, more hydrophobic, aromatic, and pharmacologically an antagonist.
- The correlation table plus the within/between variance decomposition are the main diagnostics: together they tell you whether the latent axes encode size, charge, polarity, hydrophobicity, ring/aromatic content, conformer geometry, or mostly molecule identity.
